# GreenNepal Real-World Inference
Paste these cells at the bottom of your training notebook, or run this as a standalone notebook.

**Requirements:** training notebook must have been run first so `primary_model`, `IDX_TO_CLASS`, `CLASS_NAMES`, `DEVICE`, `MODELS_DIR` etc. are all in memory.  
If running standalone, Cell 1 below reloads everything from the saved checkpoint.

## Cell A — (Standalone only) Reload model from checkpoint

In [2]:
# ── ONLY RUN THIS if you are NOT continuing from the training notebook ─────
# If primary_model is already in memory, skip this cell.

import torch, json, os
from pathlib import Path
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch.nn as nn

MODELS_DIR  = Path(os.path.expanduser('~/Downloads/Capstone_project/files/models'))
CKPT_PATH   = MODELS_DIR / 'primary_checkpoint.pth'

if not CKPT_PATH.exists():
    raise FileNotFoundError(f'Checkpoint not found: {CKPT_PATH}')

# Detect device
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
print(f'Device: {DEVICE}')

# Load checkpoint
checkpoint    = torch.load(CKPT_PATH, map_location=DEVICE)
CLASS_TO_IDX  = checkpoint['class_to_idx']
IDX_TO_CLASS  = checkpoint['idx_to_class']
CLASS_NAMES   = checkpoint['class_names']
NUM_CLASSES   = checkpoint['num_classes']
IMG_SIZE      = checkpoint['img_size']
IMAGENET_MEAN = checkpoint['imagenet_mean']
IMAGENET_STD  = checkpoint['imagenet_std']

# Rebuild model architecture (must match training)
primary_model = efficientnet_b0(weights=None)
in_features   = primary_model.classifier[1].in_features
primary_model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(256, NUM_CLASSES)
)
primary_model.load_state_dict(checkpoint['model_state_dict'])
primary_model = primary_model.to(DEVICE)
primary_model.eval()

print(f'Model loaded from {CKPT_PATH}')
print(f'Classes: {CLASS_NAMES}')
print(f'Saved test accuracy: {checkpoint["test_accuracy"]*100:.2f}%')

FileNotFoundError: Checkpoint not found: /Users/prassanna/Downloads/Capstone_project/files/models/primary_checkpoint.pth

## Cell B — Configure GreenNepal folder path

In [ ]:
import os
from pathlib import Path

# ── UPDATE THIS to wherever your GreenNepal folder is ─────────────────────
GREENNEPAL_DIR = Path(os.path.expanduser('~/Downloads/GreenNepal'))
# ───────────────────────────────────────────────────────────────────────────

if not GREENNEPAL_DIR.exists():
    raise FileNotFoundError(
        f'GreenNepal folder not found at: {GREENNEPAL_DIR}\n'
        'Update GREENNEPAL_DIR above to the correct path.'
    )

VALID_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Count images per class folder
gn_counts = {}
for cls_dir in sorted(GREENNEPAL_DIR.iterdir()):
    if cls_dir.is_dir():
        n = sum(1 for f in cls_dir.iterdir() if f.suffix.lower() in VALID_EXT)
        gn_counts[cls_dir.name] = n

print(f'GreenNepal folder: {GREENNEPAL_DIR}')
print(f'Total classes found: {len(gn_counts)}')
print()
total_imgs = 0
for cls, n in gn_counts.items():
    known = '  (unknown class — will still predict)' if cls not in CLASS_NAMES else ''
    print(f'  {cls:<16} {n:>5,} images{known}')
    total_imgs += n
print(f'\n  TOTAL            {total_imgs:>5,} images')

## Cell C — Run inference on all GreenNepal images

In [ ]:
import torch
import numpy as np
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

HIGH_CONFIDENCE   = 0.80
MEDIUM_CONFIDENCE = 0.55
HAZARDOUS_OFFSET  = 0.10

infer_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

results = []   # list of dicts, one per image

primary_model.eval()

for cls_dir in sorted(GREENNEPAL_DIR.iterdir()):
    if not cls_dir.is_dir():
        continue
    true_label = cls_dir.name
    img_files  = [f for f in cls_dir.iterdir() if f.suffix.lower() in VALID_EXT]

    for img_path in tqdm(img_files, desc=f'{true_label:<16}', leave=True):
        try:
            img    = Image.open(img_path).convert('RGB')
            tensor = infer_transform(img).unsqueeze(0).to(DEVICE)
        except Exception:
            results.append({
                'file': img_path.name, 'true': true_label,
                'pred': 'ERROR', 'confidence': 0.0,
                'tier': 'error', 'top3': [], 'correct': False
            })
            continue

        with torch.no_grad():
            logits = primary_model(tensor)
            probs  = torch.softmax(logits, dim=1).cpu().numpy()[0]

        pred_idx   = int(np.argmax(probs))
        pred_class = IDX_TO_CLASS[pred_idx]
        confidence = float(probs[pred_idx])
        top3       = [(IDX_TO_CLASS[i], float(probs[i]))
                      for i in np.argsort(probs)[::-1][:3]]

        is_haz = pred_class == 'hazardous'
        hi  = HIGH_CONFIDENCE   - (HAZARDOUS_OFFSET if is_haz else 0)
        med = MEDIUM_CONFIDENCE - (HAZARDOUS_OFFSET if is_haz else 0)
        tier = 'high' if confidence >= hi else ('medium' if confidence >= med else 'low')

        results.append({
            'file'      : img_path.name,
            'true'      : true_label,
            'pred'      : pred_class,
            'confidence': round(confidence, 4),
            'tier'      : tier,
            'top3'      : top3,
            'correct'   : pred_class == true_label,
        })

print(f'\nDone. Processed {len(results)} images.')

## Cell D — Per-class accuracy report

In [ ]:
from collections import defaultdict
import numpy as np

# Per-class breakdown
class_correct = defaultdict(int)
class_total   = defaultdict(int)
class_tiers   = defaultdict(lambda: defaultdict(int))

for r in results:
    cls = r['true']
    class_total[cls]   += 1
    class_correct[cls] += int(r['correct'])
    class_tiers[cls][r['tier']] += 1

total_correct = sum(r['correct'] for r in results)
total_images  = len(results)
overall_acc   = total_correct / total_images if total_images else 0

print(f'GreenNepal Real-World Results')
print(f'Overall accuracy: {overall_acc*100:.2f}%  ({total_correct}/{total_images})')
print()
print(f'{"Class":<16} {"Correct":>9} {"Total":>7} {"Acc":>8} {"High":>6} {"Med":>6} {"Low":>6}')
print('─' * 65)

for cls in sorted(class_total.keys()):
    tot  = class_total[cls]
    corr = class_correct[cls]
    acc  = corr / tot if tot else 0
    hi   = class_tiers[cls]['high']
    med  = class_tiers[cls]['medium']
    low  = class_tiers[cls]['low']
    flag = '  <-- check this' if acc < 0.5 else ''
    print(f'{cls:<16} {corr:>9,} {tot:>7,} {acc*100:>7.1f}% {hi:>6,} {med:>6,} {low:>6,}{flag}')

print('─' * 65)
print(f'{"TOTAL":<16} {total_correct:>9,} {total_images:>7,} {overall_acc*100:>7.1f}%')

# Save results to CSV
import csv
csv_path = MODELS_DIR / 'greennepal_results.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['file','true','pred','confidence','tier','correct'])
    writer.writeheader()
    for r in results:
        writer.writerow({k: r[k] for k in ['file','true','pred','confidence','tier','correct']})
print(f'\nDetailed results saved → {csv_path}')

## Cell E — Confusion matrix on GreenNepal data

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Only include classes that exist in GreenNepal AND in our model
gn_classes = sorted(set(r['true'] for r in results) & set(CLASS_NAMES))
cls_to_i   = {c: i for i, c in enumerate(gn_classes)}

filtered = [r for r in results if r['true'] in gn_classes and r['pred'] in gn_classes]
y_true = [cls_to_i[r['true']] for r in filtered]
y_pred = [cls_to_i.get(r['pred'], -1) for r in filtered]
valid  = [(t, p) for t, p in zip(y_true, y_pred) if p != -1]
y_true, y_pred = zip(*valid) if valid else ([], [])

if len(y_true) == 0:
    print('No matching classes between GreenNepal and model — skipping confusion matrix.')
else:
    cm      = confusion_matrix(y_true, y_pred, labels=list(range(len(gn_classes))))
    cm_norm = cm.astype('float') / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    for ax, data, fmt, title in zip(
        axes,
        [cm, cm_norm],
        ['d', '.2f'],
        ['Counts', 'Normalised']
    ):
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=gn_classes, yticklabels=gn_classes, ax=ax)
        ax.set_title(f'GreenNepal — {title}')
        ax.set_ylabel('True'); ax.set_xlabel('Predicted')
        ax.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    save_path = MODELS_DIR / 'greennepal_confusion_matrix.png'
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved → {save_path}')

    print('\nClassification Report (GreenNepal):')
    print(classification_report(y_true, y_pred, target_names=gn_classes, digits=4))

## Cell F — Visual grid of predictions (sample per class)

In [ ]:
import random, matplotlib.pyplot as plt
from PIL import Image
from collections import defaultdict

# Group results by true class
by_class = defaultdict(list)
for r in results:
    by_class[r['true']].append(r)

classes_to_show = sorted(by_class.keys())
cols = 4
rows = (len(classes_to_show) + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
axes = axes.flatten() if rows > 1 else axes

for i, cls in enumerate(classes_to_show):
    # Pick one random sample from this class
    sample = random.choice(by_class[cls])
    img_path = GREENNEPAL_DIR / cls / sample['file']

    try:
        img = Image.open(img_path).convert('RGB').resize((224, 224))
        axes[i].imshow(img)
    except Exception:
        axes[i].set_visible(False)
        continue

    correct = sample['correct']
    color   = 'green' if correct else 'red'

    # Show top-3 predictions
    top3_str = '\n'.join(
        f"  {'>' if j==0 else ' '}{cls_n}: {conf:.0%}"
        for j, (cls_n, conf) in enumerate(sample['top3'])
    )
    title = f"True: {cls}\n{top3_str}\n[{sample['tier']}]"
    axes[i].set_title(title, fontsize=7.5, color=color,
                      fontfamily='monospace', loc='left')
    axes[i].axis('off')

for j in range(len(classes_to_show), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('GreenNepal — one sample per class (green=correct, red=wrong)', fontsize=12)
plt.tight_layout()
save_path = MODELS_DIR / 'greennepal_sample_predictions.png'
plt.savefig(save_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved → {save_path}')

## Cell G — Show misclassified images (for error analysis)

In [ ]:
import random, matplotlib.pyplot as plt
from PIL import Image

wrong = [r for r in results if not r['correct'] and r['tier'] in ('high', 'medium')]
print(f'High/medium-confidence wrong predictions: {len(wrong)} / {len(results)}')

if not wrong:
    print('No high-confidence misclassifications — great result!')
else:
    show = random.sample(wrong, min(12, len(wrong)))
    cols = 4
    rows = (len(show) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = axes.flatten() if rows > 1 else [axes] * len(show)

    for i, sample in enumerate(show):
        img_path = GREENNEPAL_DIR / sample['true'] / sample['file']
        try:
            img = Image.open(img_path).convert('RGB').resize((224, 224))
            axes[i].imshow(img)
        except Exception:
            axes[i].set_visible(False)
            continue
        title = (f"True : {sample['true']}\n"
                 f"Pred : {sample['pred']} ({sample['confidence']:.0%})\n"
                 f"Tier : {sample['tier']}")
        axes[i].set_title(title, fontsize=8, color='red')
        axes[i].axis('off')

    for j in range(len(show), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('High/medium-confidence misclassifications — GreenNepal', fontsize=11)
    plt.tight_layout()
    save_path = MODELS_DIR / 'greennepal_misclassifications.png'
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved → {save_path}')